# AI Speech Detection — CNN Training on Colab GPU
**Rajib Roy | M.Tech AI | IISc**

## Setup checklist before running
1. **Runtime → Change runtime type → T4 GPU**
2. Zip your local project: right-click `voiceauth_project` → Send to → Compressed folder
3. Upload `voiceauth_project.zip` to Google Drive
4. Run cells in order

**What this notebook does:** Baselines (SVM/RF/GB) are already trained locally.
This trains CNN and CNN-LSTM on GPU (~30 and ~45 min respectively on T4).

In [ ]:
# ── CELL 1: Verify GPU ────────────────────────────────────────────────────────
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('No GPU! Go to Runtime → Change runtime type → GPU')

In [ ]:
# ── CELL 2: Mount Drive and extract project ───────────────────────────────────
from google.colab import drive
import os, zipfile

drive.mount('/content/drive')

ZIP_PATH    = '/content/drive/MyDrive/voiceauth_project.zip'  # adjust if needed
PROJECT_DIR = '/content/voiceauth_project'

if not os.path.exists(PROJECT_DIR):
    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('Done.')
else:
    print('Already extracted.')

os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ── CELL 3: Install dependencies ─────────────────────────────────────────────
!pip install -q librosa==0.10.1 soundfile pydub shap pyyaml tqdm imbalanced-learn
import librosa, torch
print(f'librosa {librosa.__version__} | torch {torch.__version__} | cuda {torch.cuda.is_available()}')

In [ ]:
# ── CELL 4: Verify feature files exist ───────────────────────────────────────
import numpy as np
from pathlib import Path

PROC = Path('data/processed')
missing = [str(PROC/f'{p}_{s}.npy')
           for s in ['train','val','test']
           for p in ['X_spectrogram','X_handcrafted','y']
           if not (PROC/f'{p}_{s}.npy').exists()]

if missing:
    print('Missing files — run Cell 4b to extract on Colab (~2h):')
    for m in missing: print(' ', m)
else:
    for split in ['train','val','test']:
        y    = np.load(PROC/f'y_{split}.npy')
        spec = np.load(PROC/f'X_spectrogram_{split}.npy', mmap_mode='r')
        print(f'{split:6s}: {len(y):5d} clips | spec {spec.shape} | genuine={(y==0).sum()} synth={(y==1).sum()}')
    print('\n✓ All feature files present. Skip Cell 4b.')

In [ ]:
# ── CELL 4b: Extract features on Colab (only if Cell 4 shows missing files) ──
# !python scripts/02_extract_features.py

In [ ]:
# ── CELL 5: Train CNN (~30 min on T4) ────────────────────────────────────────
!python scripts/04_train_cnn.py --model cnn --workers 2

In [ ]:
# ── CELL 6: Train CNN-LSTM (~45 min on T4) ───────────────────────────────────
!python scripts/04_train_cnn.py --model cnn_lstm --workers 2

In [ ]:
# ── CELL 7: Check results ─────────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

for name in ['cnn', 'cnn_lstm']:
    p = Path(f'outputs/results/{name}_test_results.csv')
    h = Path(f'outputs/results/{name}_history.csv')
    if p.exists():
        df = pd.read_csv(p)
        print(f'\n{name.upper()}:')
        print(df[['model','split','eer','f1','roc_auc','accuracy']].to_string(index=False))
    if h.exists():
        hist = pd.read_csv(h)
        best = hist.loc[hist['val_eer'].idxmin()]
        print(f'  Best epoch {int(best.epoch)} | Val EER {best.val_eer:.4f} | Val F1 {best.val_f1:.4f}')

In [ ]:
# ── CELL 8: Training curves ───────────────────────────────────────────────────
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for row, (name, col) in enumerate([('cnn','#9C27B0'),('cnn_lstm','#F44336')]):
    h = Path(f'outputs/results/{name}_history.csv')
    if not h.exists(): continue
    df = pd.read_csv(h)
    axes[row,0].plot(df.epoch, df.train_loss, color=col, lw=2, label='Train')
    axes[row,0].plot(df.epoch, df.val_loss,   color=col, lw=2, ls='--', label='Val')
    axes[row,0].set_title(f'{name.upper()} Loss'); axes[row,0].legend(); axes[row,0].grid(alpha=0.3)
    axes[row,1].plot(df.epoch, df.val_eer, color=col, lw=2)
    best_ep = df.loc[df.val_eer.idxmin(),'epoch']
    axes[row,1].axvline(best_ep, ls=':', color='gray', label=f'Best={best_ep:.0f}')
    axes[row,1].set_title(f'{name.upper()} Val EER (best={df.val_eer.min():.4f})')
    axes[row,1].legend(); axes[row,1].grid(alpha=0.3)

plt.suptitle('Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/plots/training_curves.png', dpi=150)
plt.show()

In [ ]:
# ── CELL 9: Save to Drive ─────────────────────────────────────────────────────
import shutil
DRIVE_OUT = '/content/drive/MyDrive/voiceauth_outputs'
shutil.copytree('outputs', DRIVE_OUT, dirs_exist_ok=True)
print(f'Saved to {DRIVE_OUT}')

from pathlib import Path
for f in sorted(Path(DRIVE_OUT).rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(DRIVE_OUT)}  ({f.stat().st_size/1e6:.1f} MB)')

## After the notebook finishes

1. From Drive, download `voiceauth_outputs/models/cnn_best.pt` and `cnn_lstm_best.pt`
2. Copy them to your local `outputs/models/` folder
3. Back on your laptop:
   ```bash
   python scripts/05_evaluate.py
   ```
   This combines baseline + deep model results into thesis-ready plots and tables.

## Expected performance on T4
| Model | Val EER | Test EER | Time |
|-------|---------|----------|------|
| CNN | 0.02–0.06 | 0.03–0.08 | ~30 min |
| CNN-LSTM | 0.01–0.05 | 0.02–0.07 | ~45 min |